# Problem 1: Trip Duration


In [15]:
%%duckdb -o trip_feature
SELECT date_diff('minute', start_at, stop_at) AS duration_min,
    isodow(start_at) AS day_of_week,
    hour(start_at) AS hour,
    CASE
        WHEN isodow(start_at) IN (6, 7) THEN 1
        ELSE 0
    END AS is_weekend,
    ST_Distance_Sphere(
        ST_Point("start station longitude", "start station latitude"),
        ST_Point("start station longitude", "end station latitude")
    ) +
    ST_Distance_Sphere(
        ST_Point("start station longitude", "end station latitude"),
        ST_Point("end station longitude", "end station latitude")
    ) AS distance_m,
FROM (
    SELECT
        * EXCLUDE (tripduration, starttime, stoptime),
        strptime(starttime, ['%m/%d/%Y %H:%M',
                              '%m/%d/%Y %H:%M:%S',
                              '%Y-%m-%d %H:%M:%S']) AS start_at,
        strptime(stoptime, ['%m/%d/%Y %H:%M',
                             '%m/%d/%Y %H:%M:%S',
                             '%Y-%m-%d %H:%M:%S']) AS stop_at
    FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
USING SAMPLE 100000 (reservoir, 42)


,duration_min,day_of_week,hour,is_weekend,distance_m
0,23,4,8,0,3054.828567
1,5,4,7,0,1197.845304
2,14,5,15,0,2732.864305
3,8,5,7,0,1318.940415
4,8,2,17,0,710.364293
...,...,...,...,...,...
99995,8,2,18,0,1265.152255
99996,30,2,12,0,1427.664040
99997,15,7,12,1,2129.026327
99998,9,1,5,0,2743.699010


In [16]:
%%duckdb -o ml_table_raw

SELECT *,
    (distance_m / 1000.0) / (duration_min / 60.0) AS speed_kmh
FROM trip_feature

WHERE duration_min BETWEEN 2 AND 180
  AND distance_m > 0
  AND distance_m <= 20000           
  AND duration_min IS NOT NULL
  AND distance_m IS NOT NULL
  AND day_of_week IS NOT NULL
  AND hour IS NOT NULL


,duration_min,day_of_week,hour,is_weekend,distance_m,speed_kmh
0,23,4,8,0,3054.828567,7.969118
1,5,4,7,0,1197.845304,14.374144
2,14,5,15,0,2732.864305,11.712276
3,8,5,7,0,1318.940415,9.892053
4,8,2,17,0,710.364293,5.327732
...,...,...,...,...,...,...
97396,8,2,18,0,1265.152255,9.488642
97397,30,2,12,0,1427.664040,2.855328
97398,15,7,12,1,2129.026327,8.516105
97399,9,1,5,0,2743.699010,18.291327


In [17]:
# 1-30 km/h araligini makul bir tolerans olarak aldım.
ml_table = ml_table_raw[
    (ml_table_raw["speed_kmh"] >= 1) & (ml_table_raw["speed_kmh"] <= 30)
].drop(columns=["speed_kmh"]).reset_index(drop=True)

ml_table.head()


,duration_min,day_of_week,hour,is_weekend,distance_m
0,23,4,8,0,3054.828567
1,5,4,7,0,1197.845304
2,14,5,15,0,2732.864305
3,8,5,7,0,1318.940415
4,8,2,17,0,710.364293


In [18]:
from sklearn.model_selection import train_test_split

X = ml_table.drop(columns=["duration_min"])
y = ml_table["duration_min"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)


In [19]:
import numpy as np
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.base import BaseEstimator, TransformerMixin


class CyclicalHour(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        hour = np.asarray(X).reshape(-1).astype(float)
        return np.column_stack([
            np.sin(2 * np.pi * hour / 24),
            np.cos(2 * np.pi * hour / 24),
        ])


def build_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("day_ohe", OneHotEncoder(drop="first", handle_unknown="ignore"), ["day_of_week"]),
            ("hour_cyc", CyclicalHour(), ["hour"]),
            ("weekend", "passthrough", ["is_weekend"]),
            ("distance", StandardScaler(), ["distance_m"]),
        ]
    )

    base_model = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("linreg", LinearRegression()),
    ])

    return TransformedTargetRegressor(
        regressor=base_model,
        func=np.log1p,
        inverse_func=np.expm1,
    )


model = build_model()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score

mse = mean_squared_error(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f" Validation MSE  : {mse:.2f}")
print(f" Validation MAE  : {mae:.2f}")
print(f" Validation RMSE : {rmse:.2f}")
print(f" Validation R2   : {r2:.4f}")
print(f" Y (train) ortalamasi : {np.mean(y_train):.2f} dk")


 Validation MSE  : 82.99
 Validation MAE  : 5.16
 Validation RMSE : 9.11
 Validation R2   : 0.1951
 Y (train) ortalamasi : 13.17 dk


In [10]:
final_model = build_model()
final_model.fit(X_train_val, y_train_val)
test_pred = final_model.predict(X_test)

test_mse = mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print(f" Test MSE  : {test_mse:.2f}")
print(f" Test MAE  : {test_mae:.2f}")
print(f" Test RMSE : {test_rmse:.2f}")
print(f" Test R2   : {test_r2:.4f}")


 Test MSE  : 85.37
 Test MAE  : 5.18
 Test RMSE : 9.24
 Test R2   : 0.1870


In [11]:
import pandas as pd
import duckdb

def calculate_distance_manhattan(
    start_point: tuple[float, float],
    end_point: tuple[float, float]
) -> float:
    start_lon, start_lat = start_point
    end_lon, end_lat = end_point

    result = duckdb.sql(
        """
        SELECT ST_Distance_Sphere(ST_Point(?, ?), ST_Point(?, ?))
             + ST_Distance_Sphere(ST_Point(?, ?), ST_Point(?, ?))
        """,
        params=[
            start_lon, start_lat, start_lon, end_lat,
            start_lon, end_lat, end_lon, end_lat,
        ]
    ).fetchone()

    return float(result[0])


def generate_trips(
    start_points: list[tuple[float, float]],
    end_points: list[tuple[float, float]],
    dataset: pd.DataFrame,
    number_of_trips: int = 10
) -> pd.DataFrame:

    trips: list[dict[str, float | int]] = []

    for i in range(number_of_trips):
        start_point = start_points[i]
        end_point = end_points[i]

        distance_m = calculate_distance_manhattan(start_point, end_point)
        random_row = dataset.sample(n=1, random_state=i).iloc[0]

        trips.append({
            "start_lon": start_point[0],
            "start_lat": start_point[1],
            "end_lon": end_point[0],
            "end_lat": end_point[1],
            "day_of_week": int(random_row["day_of_week"]),
            "hour": int(random_row["hour"]),
            "is_weekend": int(random_row["is_weekend"]),
            "distance_m": float(distance_m),
        })

    return pd.DataFrame(trips)


trips_df = generate_trips(
    start_points=[(-73.9772, 40.7527), (-73.9918, 40.7759), (-74.0048, 40.7198), (-73.9546, 40.7782),
                  (-73.9897, 40.7359), (-74.0134, 40.7033), (-73.9496, 40.7962), (-73.9819, 40.7684),
                  (-73.9365, 40.8047), (-74.0012, 40.7461)],
    end_points=[(-73.9498, 40.7765), (-74.0062, 40.7352), (-73.9687, 40.7584), (-73.9875, 40.7448),
                (-73.9582, 40.7196), (-73.9821, 40.7298), (-73.9714, 40.7811), (-73.9443, 40.7529),
                (-73.9657, 40.7886), (-73.9764, 40.7107)],
    dataset=ml_table,
    number_of_trips=10,
)

feature_cols = ["day_of_week", "hour", "is_weekend", "distance_m"]
trips_df["predicted_duration"] = final_model.predict(trips_df[feature_cols])
trips_df


,start_lon,start_lat,end_lon,end_lat,day_of_week,hour,is_weekend,distance_m,predicted_duration
0,-73.9772,40.7527,-73.9498,40.7765,2,18,0,3777.210761,24.461491
1,-73.9918,40.7759,-74.0062,40.7352,3,7,0,2849.263166,15.134168
2,-74.0048,40.7198,-73.9687,40.7584,2,19,0,5196.860937,40.600142
3,-73.9546,40.7782,-73.9875,40.7448,7,18,1,4684.834062,37.663378
4,-73.9897,40.7359,-73.9582,40.7196,3,17,0,4002.539836,26.874519
5,-74.0134,40.7033,-73.9821,40.7298,2,8,0,4291.949825,26.991226
6,-73.9496,40.7962,-73.9714,40.7811,1,6,0,2888.276050,15.253831
7,-73.9819,40.7684,-73.9443,40.7529,5,17,0,4656.519465,34.932238
8,-73.9365,40.8047,-73.9657,40.7886,2,8,0,3742.255342,21.959648
9,-74.0012,40.7461,-73.9764,40.7107,7,15,1,3842.546351,28.428213


In [12]:
import pydeck as pdk

mean_duration = trips_df["predicted_duration"].mean()

trips_df["color"] = trips_df["predicted_duration"].apply(
    lambda x: [255, 0, 0] if x > mean_duration else [0, 255, 0]
)

layer = pdk.Layer(
    "ArcLayer",
    data=trips_df,
    get_source_position=["start_lon", "start_lat"],
    get_target_position=["end_lon", "end_lat"],
    get_source_color="color",
    get_target_color="color",
    get_width=5,
    pickable=True,
)

view_state = pdk.ViewState(
    longitude=-73.98,
    latitude=40.75,
    zoom=10,
)

trip_map = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={"text": "Tahmini sure: {predicted_duration} dakika"},
)

trip_map


{
  "initialViewState": {
    "latitude": 40.75,
    "longitude": -73.98,
    "zoom": 10
  },
  "layers": [
    {
      "@@type": "ArcLayer",
      "data": [
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 2,
          "distance_m": 3777.210761022953,
          "end_lat": 40.7765,
          "end_lon": -73.9498,
          "hour": 18,
          "is_weekend": 0,
          "predicted_duration": 24.461490967264808,
          "start_lat": 40.7527,
          "start_lon": -73.9772
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 3,
          "distance_m": 2849.263166452015,
          "end_lat": 40.7352,
          "end_lon": -74.0062,
          "hour": 7,
          "is_weekend": 0,
          "predicted_duration": 15.134168251286548,
          "start_lat": 40.7759,
          "start_lon": -73.9918
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 2,
          "distance_m": 5196.8609369048145,
          "end_lat": 40.7584,
          "end_lon": -73.9687,
          "hour": 19,
          "is_weekend": 0,
          "predicted_duration": 40.60014240869252,
          "start_lat": 40.7198,
          "start_lon": -74.0048
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 7,
          "distance_m": 4684.83406164221,
          "end_lat": 40.7448,
          "end_lon": -73.9875,
          "hour": 18,
          "is_weekend": 1,
          "predicted_duration": 37.66337822622059,
          "start_lat": 40.7782,
          "start_lon": -73.9546
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 3,
          "distance_m": 4002.539836394438,
          "end_lat": 40.7196,
          "end_lon": -73.9582,
          "hour": 17,
          "is_weekend": 0,
          "predicted_duration": 26.874518615883073,
          "start_lat": 40.7359,
          "start_lon": -73.9897
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 2,
          "distance_m": 4291.949825079577,
          "end_lat": 40.7298,
          "end_lon": -73.9821,
          "hour": 8,
          "is_weekend": 0,
          "predicted_duration": 26.99122612024684,
          "start_lat": 40.7033,
          "start_lon": -74.0134
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 1,
          "distance_m": 2888.2760499752635,
          "end_lat": 40.7811,
          "end_lon": -73.9714,
          "hour": 6,
          "is_weekend": 0,
          "predicted_duration": 15.253830989688675,
          "start_lat": 40.7962,
          "start_lon": -73.9496
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 5,
          "distance_m": 4656.519464508957,
          "end_lat": 40.7529,
          "end_lon": -73.9443,
          "hour": 17,
          "is_weekend": 0,
          "predicted_duration": 34.93223811449366,
          "start_lat": 40.7684,
          "start_lon": -73.9819
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 2,
          "distance_m": 3742.2553418324424,
          "end_lat": 40.7886,
          "end_lon": -73.9657,
          "hour": 8,
          "is_weekend": 0,
          "predicted_duration": 21.959647949067058,
          "start_lat": 40.8047,
          "start_lon": -73.9365
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 7,
          "distance_m": 3842.5463514152007,
          "end_lat": 40.7107,
          "end_lon": -73.9764,
          "hour": 15,
          "is_weekend